In [ ]:
# Import necessary libraries
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as date
from datetime import datetime, timedelta
import time

In [ ]:
# Configure PySpark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Yahoo Finance to fetch data (data source)
import yfinance as yf

In [ ]:
# Initialize Spark Session
def create_spark_session(app_name="ML Finance Project", memory="4g"):
    """
    Create and configure a Spark session
    """
    spark = SparkSession.builder \
        .appName(app_name) \
        .config("spark.driver.memory", memory) \
        .config("spark.sql.shuffle.partitions", 4) \
        .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
        .getOrCreate()

    # Set log level to reduce verbosity
    spark.sparkContext.setLogLevel("WARN")

    return spark

# Create Spark session
spark = create_spark_session()

In [ ]:
os.makedirs('sample_data', exist_ok=True)

# Define the tickers and date range
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "META", "SPY", "QQQ", "DIA", "TSLA", "NVDA", "JPM", "BAC", "WMT", "XOM", "CVX"]

start_date = "1960-01-01"
# Use datetime.date.today() to get today's date
end_date = datetime.date.today().strftime("%Y-%m-%d")

In [ ]:
def download_and_save_stock_data():
    """
    Download stock data and save to CSV file
    """
    try:
        # Download data
        print("Downloading stock data...")
        all_stocks_df = yf.download(tickers, start=start_date, end=end_date)

        # Process the multi-index DataFrame
        processed_data = []

        # Iterate through each date in the index
        for date_idx in all_stocks_df.index:
            # Get data for this date
            date_data = all_stocks_df.loc[date_idx]

            # Process each ticker
            for ticker in tickers:
                try:
                    row_data = {
                        'Date': date_idx,
                        'Ticker': ticker,
                        'Open': date_data['Open'][ticker],
                        'High': date_data['High'][ticker],
                        'Low': date_data['Low'][ticker],
                        'Close': date_data['Close'][ticker],
                        'Volume': date_data['Volume'][ticker]
                    }
                    processed_data.append(row_data)
                except:
                    # Skip if data is not available for this ticker on this date
                    continue

        # Create DataFrame from processed data
        processed_df = pd.DataFrame(processed_data)

        # Save to CSV
        output_file = os.path.join('sample_data', 'all_stocks_data.csv')
        processed_df.to_csv(output_file, index=False)
        print(f"Data saved to {output_file}")

        return processed_df
    except Exception as e:
        print(f"Error downloading data: {e}")
        return None

In [ ]:
def process_stock_data(df):
    """
    Process the stock data into the desired format
    """
    try:
        # Convert Date to datetime if it's not already
        df['Date'] = pd.to_datetime(df['Date'])

        # Ensure numeric columns are properly typed
        numeric_columns = ['Open', 'High', 'Low', 'Close', 'Volume']
        for col in numeric_columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

        # Convert Volume to integer
        df['Volume'] = df['Volume'].fillna(0).astype(np.int64)

        # Sort by Date and Ticker
        df = df.sort_values(['Date', 'Ticker'])

        # Remove any rows where all price columns are NaN
        price_columns = ['Open', 'High', 'Low', 'Close']
        df = df.dropna(subset=price_columns, how='all')

        return df
    except Exception as e:
        print(f"Error processing data: {e}")
        print("DataFrame columns:", df.columns.tolist())
        return None


In [ ]:
# Load or download data
output_file = os.path.join('sample_data', 'all_stocks_data.csv')

In [ ]:
# Force download new data
print("Downloading fresh data from Yahoo Finance...")
all_stocks_df = download_and_save_stock_data()

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  15 of 15 completed


Data saved to sample_data/all_stocks_data.csv


In [ ]:
if all_stocks_df is not None:
  print("\nRaw data columns:", all_stocks_df.columns.tolist())

  # Process the data into the desired format
  processed_df = process_stock_data(all_stocks_df)

  if processed_df is not None:
      # Save processed data
      processed_file = os.path.join('sample_data', 'processed_stock_data.csv')
      processed_df.to_csv(processed_file, index=False)

      # Display basic information about the dataset
      print(f"\nProcessed dataset shape: {processed_df.shape}")
      print(f"\nColumns in processed dataset:")
      print(processed_df.columns.tolist())
      print(f"\nSample data for each ticker:")

      # Display sample data for each ticker
      for ticker in tickers:
          ticker_data = processed_df[processed_df['Ticker'] == ticker].head(1)
          if not ticker_data.empty:
              print(f"\n{ticker} first row:")
              with pd.option_context('display.max_columns', None, 'display.float_format', lambda x: '%.6f' % x):
                  print(ticker_data.to_string(index=False))

      # Display some basic statistics
      print("\nMissing values:")
      print(processed_df.isnull().sum())

      # Calculate and display statistics for non-null values only
      print("\nSummary statistics for numeric columns (excluding NaN):")
      with pd.option_context('display.float_format', lambda x: '%.6f' % x):
          numeric_stats = processed_df.describe(include=[np.number])
          print(numeric_stats)


Raw data columns: ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']

Processed dataset shape: (137867, 7)

Columns in processed dataset:
['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']

Sample data for each ticker:

AAPL first row:
      Date Ticker     Open     High      Low    Close    Volume
1980-12-12   AAPL 0.098726 0.099155 0.098726 0.098726 469033600

MSFT first row:
      Date Ticker     Open     High      Low    Close     Volume
1986-03-13   MSFT 0.054376 0.062373 0.054376 0.059707 1031788800

GOOGL first row:
      Date Ticker     Open     High      Low    Close    Volume
2004-08-19  GOOGL 2.490595 2.591713 2.389974 2.499063 893181924

AMZN first row:
      Date Ticker     Open     High      Low    Close     Volume
1997-05-15   AMZN 0.121875 0.125000 0.096354 0.097917 1443120000

META first row:
      Date Ticker      Open      High       Low     Close    Volume
2012-05-18   META 41.852747 44.788910 37.821746 38.050667 573576400

SPY first row:
      